<a href="https://colab.research.google.com/github/gabrielolin/16886-Embodied-AI-Safety/blob/main/%5BS26_EAISafety%5D%5BHW3%5D_Uncertainty_Quantification_in_Robotics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Conformal Prediction

Conformal prediction (CP) is statistical technique for generating prediction sets (or intervals) for any black-box model such that the true output of the model is contained within the sets with high probability. While there are many variants of conformal prediction, in this homework we focus on *split conformal prediction*.

Split conformal prediction takes a pre-trained predictive model $f: X \rightarrow Y$ which maps from inputs $X$ to prediction labels $Y$ (e.g. a neural network classifier), a small amount of additional i.i.d. calibration data $D_{calib}= \{(x_i, y_i)\}^n_{i=1}$, a nonconformity score function $s:X \times Y \rightarrow \mathbb{R}$ which captures a heuristic notion of uncertainty, and a desired coverage level $\alpha \in [0,1]$.

Given $f$, $D_{calib}$, $s$, and $\alpha$, the conformal prediction process allows us to construct prediction sets $C(x_{n+1})$ for any fresh test datapoint we see $x_{n+1}$ with the coverage guarantee that $$P(Y_{n+1} \in C(x_{n+1})) \geq 1-\alpha$$
meaning the true output is contained within set $C(x_{n+1})$ with at least $1-\alpha$ probability.

When $\alpha=0.1$ for example, CP gives us the guarantee that our sets $C(x_{n+1})$ contain $y_{n+1}$ with 90% confidence!

### Exercise 1: Split Conformal on Image Classifiers

Import basic dependencies.

In [ ]:
# for directories and paths
import os

# for loading images and pretrained classifiers
import torch
import torchvision
from PIL import Image
from torchvision import transforms
from torch import Tensor

# for math operations
import numpy as np
import scipy

# for plotting
import matplotlib.pyplot as plt

Load in a pretrained image classifier, $f$. We will demonstrate our example using ResNet-18. ResNet-18 is a convolutional neural network that is 18 layers deep. You can load a pretrained version of the network trained on more than a million images from the ImageNet database [1]. The pretrained network can classify images into 1000 object categories, such as keyboard, mouse, pencil, and many animals.

[1] Deng, Jia, et al. "Imagenet: A large-scale hierarchical image database." 2009 IEEE conference on computer vision and pattern recognition. Ieee, 2009.

In [ ]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
# or any of these variants
# model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet34', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet101', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet152', pretrained=True)
model.eval()

Load in ImageNet classes.

In [ ]:
!wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt


In [ ]:
# Read the categories
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

In [ ]:
# Print 10 of imagenet classes

print(categories[:10])

# Load $D_{calib}$

For calibration and testing, we will use Imagenette (https://github.com/fastai/imagenette?tab=readme-ov-file#imagenette-1).

> Imagenette is a subset of 10 classes from Imagenet (tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute).

For the purpose of this exercise, we will assume that given our pretrained off-the-shelf ResNet-18 classifier, what we care to do with the model downstream is to use the model on Imagenette (our test distribution).

Assume all images in Imagenette are unseen by our model during training.
From Imagenette, we will sample $D_{calib}$ our calibration dataset.




In [ ]:
imagenette_dataset = torchvision.datasets.Imagenette(root="./", download=True)

In [ ]:
# Confirm the calibration classes
list_of_classes = imagenette_dataset.classes
print("list_of_classes", list_of_classes)

# Test classifier.

Test passing one of the images in $D_{calib}$ to classifier $f$. What are the top 5 output classes predicted by $f$ for your image?

In [ ]:
# Ensure that images are preprocessed as expected for input to the network
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# randomly pick an image
random_img = imagenette_dataset[np.random.choice(len(imagenette_dataset))]
img_tensor = preprocess(random_img[0])
img_batch = img_tensor.unsqueeze(0) # create a mini-batch as expected by the model

# move the input and model to GPU for speed if available
if torch.cuda.is_available():
    img_batch = img_batch.to('cuda')
    model.to('cuda')

with torch.no_grad():
    output = model(img_batch) # Tensor of shape 1000, with confidence scores over ImageNet's 1000 classes

# The output has unnormalized scores. To get probabilities, you can run a softmax on it.
probabilities = torch.nn.functional.softmax(output[0], dim=0)

# Show top 5 categories per image
top5_prob, top5_catid = torch.topk(probabilities, 5)
for i in range(top5_prob.size(0)):
    print(categories[top5_catid[i]], top5_prob[i].item())

# Show image
random_img[0]

# TODO 1.1: Implement Split Conformal Prediction

Now that we have taken a look at our classifier $f$, let's run conformal prediction.

Let's specify to our model:
1. our calibration dataset $D_{calib}$
2. our desired coverage threshold, $\alpha$,
3. our nonconformity score function $s$

In [ ]:
# TODO: Set alpha such that our desired coverage is 90%.
alpha = None

# Choose the calibration dataset size
N_calib = 100
N_test = 100

D_calib_indices = np.random.choice(len(imagenette_dataset), size=N_calib)
D_calib_X = [preprocess(imagenette_dataset[i][0]) for i in D_calib_indices]
D_calib_Y = [categories.index(list_of_classes[imagenette_dataset[i][1]][0]) for i in D_calib_indices]

# use the rest for test. Take the first 100 to make this run faster.
D_test_indices = np.random.choice(np.setdiff1d(np.arange(len(imagenette_dataset)), D_calib_indices),size=N_test)
D_test_X = [preprocess(imagenette_dataset[i][0]) for i in D_test_indices]
D_test_Y = [categories.index(list_of_classes[imagenette_dataset[i][1]][0]) for i in D_test_indices]



# Conformal Prediction Algorithm

1. For each $\{(x_i, y_i)\} \in D_{calib}$, compute the nonconformity score $s(x_i, y_i) = \sum_{j=1}^k f(x_i)_{(j)}$ (the amount of estimated probability you need to include in the prediction set to include the true label).
2. Take $q$  the $(1-\alpha)$th empirical quantile of the set of scores.
3. For each new datapoint seen at test, construct prediction sets $C(x_{test}) = \{y: s(x_{test}, y) \leq q\}$.

## We define a few helper functions. You will want to fill out the function conformal_prediction().

In [ ]:
def get_predictions(model, D_calib_inputs):

    input_batch = torch.stack(D_calib_inputs, dim=0)

    # move the input and model to GPU for speed if available
    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda')
        model.to('cuda')

    with torch.no_grad():
        output = model(input_batch)
    # Tensor of shape 1000, with confidence scores over ImageNet's 1000 classes
    # The output has unnormalized scores. To get probabilities, you can run a softmax on it.
    probabilities = torch.nn.functional.softmax(output, dim=1)
    return probabilities

In [ ]:
def compute_nonconformity_scores(softmax_probabilities:Tensor, D_calib_Y:list):
  sorted_idxs = softmax_probabilities.argsort(1, descending=True).detach().numpy()
  sorted_softmax_values = np.take_along_axis(softmax_probabilities, sorted_idxs, axis=1).cumsum(axis=1)
  cal_scores = []
  for i in range(len(sorted_softmax_values)):
      true_idx = D_calib_Y[i]
      total = #TODO hint: use sorted_softmax_values
      cal_scores.append(total)
  return np.array(cal_scores)



In [ ]:
def compute_quantile(nonconformity_scores:Tensor, alpha:float):
  # Get the score quantile. Make sure to use a finite sample correction!
  # Source: Equation 7 https://www.stat.berkeley.edu/~ryantibs/statlearn-s23/lectures/conformal.pdf
  # note: anushri's lecture has something slightly different for pedagogical reasons. use the
  # equation in the paper for the final implementation
  qhat = np.quantile('''TODO''')
  return qhat

In [ ]:
def conformal_prediction(model, D_calib_X: Tensor, D_calib_Y: list, alpha: float) -> float:

  softmax_probabilities = get_predictions(model, D_calib_X)
  # print("softmax_probabilities", softmax_probabilities.shape)
  nonconformity_scores = compute_nonconformity_scores(softmax_probabilities, D_calib_Y) # TODO: you will need to write this function
  q = compute_quantile(nonconformity_scores, alpha) # TODO: Write this function

  return q



In [ ]:
qhat = conformal_prediction(model, D_calib_X, D_calib_Y, alpha)

In [ ]:
print("qhat ", qhat)

# TODO 1.2: Construct Prediction Intervals Using $q$ obtained from the Conformal Prediction Procedure.

For each datapoint in your test set, construct the calibrated prediction interval using $q$ and measure coverage achieved on $D_{test}$.


In [ ]:
def construct_prediction_intervals(model, D_test_X:Tensor, D_test_Y:list, q: float) -> list:

  test_probabilities = get_predictions(model, D_test_X)
  # TODO: make prediction set
  # uncomment the answer below
  sorted_idxs = test_probabilities.argsort(1, descending=True).detach().numpy()
  sorted_softmax_values = np.take_along_axis(test_probabilities, sorted_idxs, axis=1).cumsum(axis=1)
  covereds = []
  sets = []
  for i in range(len(sorted_softmax_values)):
      true_idx = D_test_Y[i]
      prediction_set = []
      total_prob, j = 0, 0
      while total_prob < qhat:
          # TODO: add to the prediction set, add to the total probability, and increment j

      sets.append(prediction_set)
      covereds.append(1 if true_idx in prediction_set else 0)


  return sets, covereds


In [ ]:
sets, covereds = construct_prediction_intervals(model, D_test_X, D_test_Y, qhat)

# Take a look at our conformalized prediction sets!

In [ ]:
i= np.random.choice(len(D_test_indices))
dataset_idx = D_test_indices[i]

print("true class: ",list_of_classes[imagenette_dataset[dataset_idx][1]][0])
print("prediction set: ", [categories[sets[i][j]] for j in range(len(sets[i]))])
print("\n\n\n")

imagenette_dataset[dataset_idx][0]



In [ ]:
coverage_percent = np.mean(covereds)
print("coverage_percent", coverage_percent)
plt.clf()
result = plt.hist(covereds, bins=2, color='c', edgecolor='k', alpha=0.65)
plt.axvline(np.mean(covereds), color='k', linestyle='dashed', linewidth=1)
plt.show()


# TODO 1.3: Short Answer

1. Did you reach the desired coverage level? Report the mean and standard deviation for average over 10 runs.
2. How did the choice of N_calib affect the achieved mean coverage?


Write your response in the cell below.

---



# 2. Uncertainty Quantification for Robotic Failure Detection

In [ ]:
!git clone https://github.com/junwon-vision/eais_uq.git

import os
import sys
from pathlib import Path

EAIS_UQ_DIR = Path("eais_uq").resolve()
if str(EAIS_UQ_DIR) not in sys.path:
    sys.path.insert(0, str(EAIS_UQ_DIR))

CKPT_ZIP = Path("model_checkpoints.zip")

import gdown
gdown.download(id="1d9W43ejh3Fv98tmnwhCBqZVYms86U_0U", output=str(CKPT_ZIP), quiet=False)
!unzip -q -o {CKPT_ZIP} -d .
print("Checkpoints extracted.")

!mv data eais_uq/data

In [ ]:
import sys, os, pickle, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import Video

from config import *
from data_utils import *
from models import *
from uq_inference import *
from density import *
from conformal import *
from visualization import *

print(f"Device: {DEVICE}")
print(f"Data dir: {DATA_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"OOD dir: {OOD_DIR}")

print("\nLoading models...")
baseline_model, _  = load_classifier(CLASSIFIER_CHECKPOINTS["baseline"],   DEVICE, mode="baseline")
mc_model,       _  = load_classifier(CLASSIFIER_CHECKPOINTS["mc_dropout"], DEVICE, mode="mc_dropout")
ensemble_model, _  = load_classifier(CLASSIFIER_CHECKPOINTS["ensemble"],   DEVICE, mode="ensemble")
density_model, dino_encoder = load_density_model(DENSITY_CHECKPOINT, DEVICE)
print("All models loaded.")

eval_episodes = load_episodes(["failure_eval.pkl"])
cal_episodes  = load_episodes(["calibration_combined.pkl"])
print(f"Eval: {len(eval_episodes)}, Cal: {len(cal_episodes)}")

OOD_EP_INDICES = [0, 1, 2, 8, 10]

def load_ood_episode(idx, ood_dir=OOD_DIR):
    pkl = sorted(ood_dir.glob("traj_*.pkl"))[idx]
    with open(pkl, "rb") as f:
        data = pickle.load(f)
    traj = data[0]
    front_imgs, wrist_imgs, labels = [], [], []
    for info, _, label in traj:
        front_imgs.append(info["cam_rs"][0])
        wrist_imgs.append(info["cam_zed_right"][0])
        labels.append(label)
    return {"front_cam": front_imgs, "wrist_cam": wrist_imgs,
            "failure": np.array(labels, dtype=np.float32)}

ood_episodes = [load_ood_episode(i) for i in OOD_EP_INDICES]
print(f"OOD episodes loaded: {len(ood_episodes)}")

## 2.1 What is Uncertainty Quantification?

A predictive model produces predictions $\hat{y} = f_\theta(x)$, but how confident should we be in that prediction?

**Uncertainty Quantification (UQ)** answers this question by treating the prediction as a random variable and characterizing its distribution.

### Bayesian Formulation

Given training data $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^N$, the **Bayesian predictive distribution** is:

$$p(y \mid x, \mathcal{D}) = \int p(y \mid x, \theta)\, p(\theta \mid \mathcal{D})\, d\theta$$

where:
- $p(\theta \mid \mathcal{D})$ is the **posterior** over model parameters
- $p(y \mid x, \theta)$ is the **likelihood** (model prediction for fixed parameters)


## 2.2 UQ Methods

We use four practical approaches to quantify uncertainty for the robotic failure classifier. For each, we run inference on an evaluation trajectory and an OOD trajectory.

### Entropy (Baseline)

The simplest UQ measure: the **binary entropy** of the model's output probability.

$$H(p) = -p \log p - (1-p) \log(1-p)$$

where $p = \sigma(f_\theta(x))$ is the predicted failure probability. $H$ is maximized at $p = 0.5$ (maximum uncertainty) and minimized at $p \in \{0, 1\}$ (full confidence).

### Monte Carlo Dropout

Keep dropout **enabled** at test time and run $T$ forward passes (Gal & Ghahramani, 2016):

$$\bar{p} = \frac{1}{T} \sum_{t=1}^{T} \sigma\!\big(f_{\theta_t}(x)\big), \quad \text{Var}[p] = \frac{1}{T} \sum_{t=1}^{T} \big(\sigma(f_{\theta_t}(x)) - \bar{p}\big)^2$$

### Deep Ensemble

Train $K$ independent models with different random seeds and aggregate (Lakshminarayanan et al., 2017):

$$\bar{p} = \frac{1}{K} \sum_{k=1}^{K} \sigma\!\big(f_{\theta_k}(x)\big), \quad \text{Var}[p] = \frac{1}{K} \sum_{k=1}^{K} \big(\sigma(f_{\theta_k}(x)) - \bar{p}\big)^2$$

### Density Estimation (Flow Matching)

Instead of modeling $p(\theta \mid \mathcal{D})$, model the **empirical feature distribution** $p(z)$ where $z = \text{DINOv2}(x)$. Points with low feature density are likely OOD (Lipman et al., 2023; following [FailDetect, arXiv:2503.08558](https://arxiv.org/abs/2503.08558)):

$$z = x + v_\theta(x, t{=}0), \quad \text{logpZO} = \|z\|^2$$

Higher logpZO → feature is far from training distribution → more uncertain.

**TODO 2.1**
What is the difference between the UQ methods? How are the train time / test time efficiency of each method different?



**TODO 2.2**
Run inference on an evaluation trajectory. (Run the code below.)

It will run UQ for 1) normal data with the orange block and 2) OOD data with the blue block.

Discuss the performance of the UQ methods (e.g., if uncertainty is aligned with the inaccuracy of the model). How does the estimated uncertainty different for normal and OOD data?


In [ ]:
# Select one eval trajectory and one OOD trajectory to use throughout this section
EVAL_IDX = 1
OOD_IDX  = 0  # index into ood_episodes list

ep_eval = eval_episodes[EVAL_IDX]
ep_ood  = ood_episodes[OOD_IDX]
ood_ep_raw_idx = OOD_EP_INDICES[OOD_IDX]

# Run all 4 UQ methods on the eval episode
base_probs_eval, _, base_ents_eval = run_inference_episode(baseline_model, ep_eval, mode="baseline",   device=DEVICE)
mc_probs_eval,   mc_vars_eval,   _ = run_inference_episode(mc_model,       ep_eval, mode="mc_dropout", device=DEVICE)
ens_probs_eval,  ens_vars_eval,  _ = run_inference_episode(ensemble_model, ep_eval, mode="ensemble",   device=DEVICE)
dens_eval = predict_density_episode(density_model, dino_encoder, ep_eval, DEVICE)

# Run all 4 UQ methods on the OOD episode
base_probs_ood, _, base_ents_ood = run_inference_episode(baseline_model, ep_ood, mode="baseline",   device=DEVICE)
mc_probs_ood,   mc_vars_ood,   _ = run_inference_episode(mc_model,       ep_ood, mode="mc_dropout", device=DEVICE)
ens_probs_ood,  ens_vars_ood,  _ = run_inference_episode(ensemble_model, ep_ood, mode="ensemble",   device=DEVICE)
dens_ood = predict_density_episode(density_model, dino_encoder, ep_ood, DEVICE)

print("Inference complete.")
print(f"Eval ep {EVAL_IDX}:  {len(ep_eval['failure'])} timesteps")
print(f"OOD ep {ood_ep_raw_idx}: {len(ep_ood['failure'])} timesteps")

# UQ videos — eval episode
(OUTPUT_DIR / "videos").mkdir(parents=True, exist_ok=True)

clf_methods_eval = [
    ("baseline",   base_probs_eval, base_ents_eval, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs_eval,   mc_vars_eval,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs_eval,  ens_vars_eval,  "Variance", MODE_UQ_YMAX["ensemble"]),
]
for mode, probs, uq, label, ymax in clf_methods_eval:
    vp = render_episode_video(
        ep_eval, EVAL_IDX, probs, uq,
        uq_label=label, mode=mode, fps=10, uq_ymax=ymax,
        output_path=OUTPUT_DIR / "videos" / f"{mode}_eval_ep{EVAL_IDX}.mp4")
    print(f"[eval][{mode}] {vp}")
    display_video(vp)

vp = render_episode_video_density(
    ep_eval, EVAL_IDX, ens_probs_eval, dens_eval, ens_vars_eval,
    uq_label="Variance", mode="ensemble", fps=10,
    uq_ymax=MODE_UQ_YMAX["ensemble"], logpZO_ymax=DENSITY_YMAX,
    output_path=OUTPUT_DIR / "videos" / f"density_eval_ep{EVAL_IDX}.mp4")
print(f"[eval][density] {vp}")
display_video(vp)

# UQ videos — OOD episode
clf_methods_ood = [
    ("baseline",   base_probs_ood, base_ents_ood, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs_ood,   mc_vars_ood,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs_ood,  ens_vars_ood,  "Variance", MODE_UQ_YMAX["ensemble"]),
]
for mode, probs, uq, label, ymax in clf_methods_ood:
    vp = render_episode_video(
        ep_ood, ood_ep_raw_idx, probs, uq,
        uq_label=label, mode=mode, fps=10, uq_ymax=ymax,
        output_path=OUTPUT_DIR / "videos" / f"{mode}_ood{ood_ep_raw_idx}.mp4")
    print(f"[OOD][{mode}] {vp}")
    display_video(vp)

vp = render_episode_video_density(
    ep_ood, ood_ep_raw_idx, ens_probs_ood, dens_ood, ens_vars_ood,
    uq_label="Variance", mode="ensemble", fps=10,
    uq_ymax=MODE_UQ_YMAX["ensemble"], logpZO_ymax=DENSITY_YMAX,
    output_path=OUTPUT_DIR / "videos" / f"density_ood{ood_ep_raw_idx}.mp4")
print(f"[OOD][density] {vp}")
display_video(vp)

**TODO 2.3**
How does the uncertainty measured by density estimation differ between normal and OOD data?

**TODO 2.4**
Discuss the limitation of the UQ methods. (Provide at least one limitation for each method.) And discuss at least two ways to improve the performance of the UQ methods.

## 2.3 Conformal Calibration for Uncertainty Thresholding

The UQ methods above produce a **scalar score** that reflects model uncertainty. To use this score in practice, we need a principled way to choose a **threshold** for deciding whether a prediction should be trusted or flagged. Selecting such a threshold manually provides no statistical guarantee.

**Conformal prediction** provides a distribution-free framework for calibrating this threshold using a held-out calibration dataset.

### Guarantee

When calibration data is in-distribution:
$$\Pr\!\left(s(X) \le \hat{q} \mid X \in \mathcal{D}_{\mathrm{in}}\right) \ge 1-\alpha$$

This guarantees that at least $1-\alpha$ of future in-distribution samples will have uncertainty below the calibrated threshold. Note this is a **recall guarantee on in-distribution data**, not a precision guarantee on OOD detection.

**TODO: 2.5.**
What is (an) assumption(s) of the conformal calibration? Discuss the limitation of the conformal calibration for calibrating the OOD detection threshold.

In [ ]:
alpha = 0.1
CACHE_FILE = Path("eais_uq/output/conformal_thresholds.json")
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

classifier_cfg = [
    ("baseline",   "entropies"),
    ("mc_dropout", "variances"),
    ("ensemble",   "variances"),
]

if CACHE_FILE.exists():
    print(f"Loading cached thresholds from {CACHE_FILE}")
    with open(CACHE_FILE) as f:
        all_thresholds = json.load(f)
    all_cal_results = {}
else:
    model_objs = {"baseline": baseline_model, "mc_dropout": mc_model, "ensemble": ensemble_model}
    all_thresholds = {}
    all_cal_results = {}

    for mode, uq_metric in classifier_cfg:
        thr, cal_res = run_calibration(model_objs[mode], cal_episodes, mode=mode,
                                       device=DEVICE, alpha=alpha,
                                       uq_metric=uq_metric, batch_size=32)
        all_thresholds[mode] = {
            k: {"threshold": float(v["threshold"]),
                "n_samples": int(v["n_samples"]),
                "description": v["description"]}
            for k, v in thr.items() if k in ("image_id", "traj_id")
        }
        all_cal_results[mode] = cal_res

    dens_thr, dens_res = run_calibration_density(
        density_model, dino_encoder, cal_episodes, DEVICE, alpha=0.005)
    all_thresholds["density"] = {
        k: {"threshold": float(v["threshold"]),
            "n_samples": int(v["n_samples"]),
            "description": v["description"]}
        for k, v in dens_thr.items()
    }
    all_cal_results["density"] = dens_res

    with open(CACHE_FILE, "w") as f:
        _json_dump(all_thresholds, f, indent=2)
    print(f"Thresholds saved to {CACHE_FILE}")

print(f"\nConformal Calibration (alpha={alpha}, target coverage >= {1-alpha:.0%})")
print("=" * 70)
for mode, _ in classifier_cfg + [("density", "logpZO")]:
    print(f"\n  [{mode}]")
    for cal_key in ["image_id", "traj_id"]:
        info = all_thresholds[mode][cal_key]
        print(f"    {cal_key:12s} | tau={info['threshold']:.6f}  (n={info['n_samples']:5d})")

**TODO 2.6 Visualize the image-level calibration threshold. (Run the code below.)**

This will visualize the threshold calibrated with taking each image as an independent sample, and computing empirical quantiles over the uncertainty scores of all images in the calibration set.


How does the threshold look like? Is it overly-conservative or overly-restrictive?
What is the limitation of the image-level threshold?

### Visualizing the Calibrated Threshold (Image-Level)

We now overlay the calibrated threshold $\hat{q}$ on the same trajectories used above. At each timestep, a UQ score **above** the threshold triggers a flag (the model is uncertain about this observation).

The image-level threshold is computed from individual timesteps in the calibration set — each image is treated as an independent sample.

In [ ]:
# Image-level threshold — eval episode
clf_methods_eval_cp = [
    ("baseline",   base_probs_eval, base_ents_eval, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs_eval,   mc_vars_eval,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs_eval,  ens_vars_eval,  "Variance", MODE_UQ_YMAX["ensemble"]),
]
print("=== Eval episode (image-level threshold) ===")
for mode, probs, uq, label, ymax in clf_methods_eval_cp:
    tau = all_thresholds[mode]["image_id"]["threshold"]
    vp = render_episode_video(
        ep_eval, EVAL_IDX, probs, uq,
        uq_label=label, mode=mode, fps=10,
        threshold=tau, uq_ymax=ymax,
        output_path=OUTPUT_DIR / "videos" / f"{mode}_imageid_eval_ep{EVAL_IDX}.mp4")
    print(f"[eval][{mode}] tau={tau:.6f} -> {vp}")
    display_video(vp)

tau_dens = all_thresholds["density"]["image_id"]["threshold"]
vp = render_episode_video_density(
    ep_eval, EVAL_IDX, ens_probs_eval, dens_eval,
    classifier_uq=dens_eval,
    uq_label="logpZO", mode="density", fps=10,
    threshold=tau_dens, logpZO_ymax=DENSITY_YMAX,
    output_path=OUTPUT_DIR / "videos" / f"density_imageid_eval_ep{EVAL_IDX}.mp4")
print(f"[eval][density] tau={tau_dens:.6f} -> {vp}")
display_video(vp)

# Image-level threshold — OOD episode
clf_methods_ood_cp = [
    ("baseline",   base_probs_ood, base_ents_ood, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs_ood,   mc_vars_ood,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs_ood,  ens_vars_ood,  "Variance", MODE_UQ_YMAX["ensemble"]),
]
print("\n=== OOD episode (image-level threshold) ===")
for mode, probs, uq, label, ymax in clf_methods_ood_cp:
    tau = all_thresholds[mode]["image_id"]["threshold"]
    vp = render_episode_video(
        ep_ood, ood_ep_raw_idx, probs, uq,
        uq_label=label, mode=mode, fps=10,
        threshold=tau, uq_ymax=ymax,
        output_path=OUTPUT_DIR / "videos" / f"{mode}_imageid_ood{ood_ep_raw_idx}.mp4")
    print(f"[OOD][{mode}] tau={tau:.6f} -> {vp}")
    display_video(vp)

vp = render_episode_video_density(
    ep_ood, ood_ep_raw_idx, ens_probs_ood, dens_ood,
    classifier_uq=dens_ood,
    uq_label="logpZO", mode="density", fps=10,
    threshold=tau_dens, logpZO_ymax=DENSITY_YMAX,
    output_path=OUTPUT_DIR / "videos" / f"density_imageid_ood{ood_ep_raw_idx}.mp4")
print(f"[OOD][density] tau={tau_dens:.6f} -> {vp}")
display_video(vp)

## 2.4 Trajectory-Level Conformal Prediction

### Key Insight

While timesteps within a trajectory are temporally dependent (violating exchangeability), **trajectories themselves are i.i.d.** — each episode is an independent rollout.

We restore the exchangeability assumption by working at the trajectory level. Define the trajectory-level nonconformity score as:

$$S = \max_{t} s_t$$

where $s_t$ is the per-timestep UQ score. Using $S$ as the nonconformity score, the conformal guarantee becomes:

$$P\!\big(\max_t u_t \le \hat{q}\big) \ge 1 - \alpha$$

This guarantees that the UQ stays below threshold for the **entire trajectory** with probability $\ge 1 - \alpha$.

### From Trajectory-Level to Image-Level

Because $S = \max_t s_t$, if a trajectory satisfies $\max_t u_t < \hat{q}$, then every individual timestep also satisfies $u_t < \hat{q}$. The same threshold $\hat{q}$ can therefore be applied per-image at test time — the formal guarantee is still **trajectory-wise**.

**References:**
- [KnowNo (Ren et al., 2023)](https://arxiv.org/abs/2307.01928)
- [UniSafe (Seo et al., 2025)](https://arxiv.org/abs/2505.00779)

**TODO 2.7** Visualize the trajectory-level calibration threshold. (Run the code below.)

This will visualize the threshold calibrated with taking the maximum uncertainty seen across all timesteps in a trajectory, and computing empirical quantiles over the uncertainty scores of all trajectories in the calibration set.

How does the threshold look like compared to the image-level threshold?

In [ ]:
# Compare image-level vs trajectory-level thresholds
print("Threshold comparison:")
for mode in ["baseline", "mc_dropout", "ensemble", "density"]:
    img_tau  = all_thresholds[mode]["image_id"]["threshold"]
    traj_tau = all_thresholds[mode]["traj_id"]["threshold"]
    print(f"  [{mode:12s}]  image_id={img_tau:.6f}  traj_id={traj_tau:.6f}  "
          f"ratio={traj_tau/(img_tau+1e-10):.2f}x")

# Trajectory-level threshold — eval episode
print("\n=== Eval episode (trajectory-level threshold) ===")
for mode, probs, uq, label, ymax in clf_methods_eval_cp:
    tau = all_thresholds[mode]["traj_id"]["threshold"]
    vp = render_episode_video(
        ep_eval, EVAL_IDX, probs, uq,
        uq_label=label, mode=mode, fps=10,
        threshold=tau, uq_ymax=ymax,
        output_path=OUTPUT_DIR / "videos" / f"{mode}_trajid_eval_ep{EVAL_IDX}.mp4")
    print(f"[eval][{mode}] tau={tau:.6f} -> {vp}")
    display_video(vp)

tau_dens_traj = all_thresholds["density"]["traj_id"]["threshold"]
vp = render_episode_video_density(
    ep_eval, EVAL_IDX, ens_probs_eval, dens_eval,
    classifier_uq=dens_eval,
    uq_label="logpZO", mode="density", fps=10,
    threshold=tau_dens_traj, logpZO_ymax=DENSITY_YMAX,
    output_path=OUTPUT_DIR / "videos" / f"density_trajid_eval_ep{EVAL_IDX}.mp4")
print(f"[eval][density] tau={tau_dens_traj:.6f} -> {vp}")
display_video(vp)

# Trajectory-level threshold — OOD episode
print("\n=== OOD episode (trajectory-level threshold) ===")
for mode, probs, uq, label, ymax in clf_methods_ood_cp:
    tau = all_thresholds[mode]["traj_id"]["threshold"]
    vp = render_episode_video(
        ep_ood, ood_ep_raw_idx, probs, uq,
        uq_label=label, mode=mode, fps=10,
        threshold=tau, uq_ymax=ymax,
        output_path=OUTPUT_DIR / "videos" / f"{mode}_trajid_ood{ood_ep_raw_idx}.mp4")
    print(f"[OOD][{mode}] tau={tau:.6f} -> {vp}")
    display_video(vp)

vp = render_episode_video_density(
    ep_ood, ood_ep_raw_idx, ens_probs_ood, dens_ood,
    classifier_uq=dens_ood,
    uq_label="logpZO", mode="density", fps=10,
    threshold=tau_dens_traj, logpZO_ymax=DENSITY_YMAX,
    output_path=OUTPUT_DIR / "videos" / f"density_trajid_ood{ood_ep_raw_idx}.mp4")
print(f"[OOD][density] tau={tau_dens_traj:.6f} -> {vp}")
display_video(vp)

# 3. Conformal Verification of Safe Sets

In homework 2 we used learned-based techniques to approximate Hamilton-Jacobi value functions. A natural question then, is in what circumstances can we trust the approximated solution? Lin et al. [1] showed that you can use conformal prediction to generate a probabilistic guarantee on the quality of the learned value function.

[1] Albert Lin and Somil Bansal, Verification of neural reachable tubes via scenario optimization and conformal prediction, L4DC 2024

In [ ]:
!git clone https://github.com/kensukenk/eais_hw3
!pip install gdown
!gdown 1xmZEntbsMMEYuCapJfYw72GihKgiGA97


In [ ]:
!pip install configargparse==1.7
!pip install scikit-learn tensorboard
!pip install plotly

In [ ]:
!python eais_hw3/deepreach/run_experiment.py --mode train --experiment_class DeepReach --dynamics_class Dubins3D --experiment_name dubins3d_exact_run --minWith target --goalR 0.5 --velocity 1. --omega_max 1.25 --angle_alpha_factor 1.2 --set_mode avoid --deepreach_model exact --pretrain

In [ ]:
!cd eais_hw3/deepreach
import sys
sys.path.append('eais_hw3/deepreach')
from utils import modules, dataio, losses, diff_operators
from dynamics import dynamics
from experiments import experiments
import inspect

import pickle
import torch
model = modules.SingleBVPNet(in_features=4, out_features=1, type='sine', mode='mlp',
                             final_layer_factor=1., hidden_features=512, num_hidden_layers=3)

checkpoint = torch.load('runs/dubins3d_exact_run/training/checkpoints/model_current.pth')
device = 'cpu'
model_state_dict = model.state_dict()
for key in checkpoint.keys():
    if key in model_state_dict:
        model_state_dict[key] = checkpoint[key]

# Now load the updated state_dict into the model
model.load_state_dict(model_state_dict)
model.to(device)

with open('runs/dubins3d_exact_run/orig_opt.pickle', 'rb') as opt_file:
    orig_opt = pickle.load(opt_file)


dynamics_class = getattr(dynamics, orig_opt.dynamics_class)
dynamics = dynamics_class(**{argname: getattr(orig_opt, argname) for argname in inspect.signature(dynamics_class).parameters.keys() if argname != 'self'})
dynamics.deepreach_model=orig_opt.deepreach_model
dataset = dataio.ReachabilityDataset(
    dynamics=dynamics, numpoints=orig_opt.numpoints,
    pretrain=orig_opt.pretrain, pretrain_iters=orig_opt.pretrain_iters,
    tMin=orig_opt.tMin, tMax=orig_opt.tMax,
    counter_start=orig_opt.counter_start, counter_end=orig_opt.counter_end,
    num_src_samples=orig_opt.num_src_samples, num_target_samples=orig_opt.num_target_samples)


To use our value function as a safety monitor, we typically override a nominal policy when the value function is at the boundary of the zero-sublevel set. However, if the value function has positive value (indicating safety) for states that are actually unsafe then our safety monitor will optimistically allow a nominal policy to take an unsafe action.

Then, a natural question is to ask "how accurate is the safe set $\mathcal{S}$ induced by the zero-superlevel set of our learned value function?" It turns out, we can analyze this with conformal prediction! To do so, we follow the analysis from [1] and first define the following nonconformity score:

$-J_{\pi}(x) := -\min_{\tau \in[0,T]} l(\xi^\pi_x(\tau))$,

**TODO** What does this non-conformity score measure? Under which cases is this score positive?

Then, by Theorem 3 in [1], we can select $N$ calibration points from the zero-super level set of the learned value function $\{x \; | \;  V(x,0) > 0 \}$. For each of these $N$ points, we can simulate the evolution of the state by following the safety policy $\pi = argmax_u \nabla_x V(x,0) \cdot f(x,u)$ and count all $k$ outliers where $J_{\pi}(x) < 0$.


Let's first generate our calibration sets and calculate our nonconformity scores.

In [ ]:
# size of calibration set
N = 10000

# Get N points in the zero superlevel set
num_pos = 0
calib_points = torch.zeros(N, 4)
neg_mask = torch.ones(N, dtype=torch.bool)
while num_pos < N:
  num_samples = N - num_pos
  calib_points[neg_mask, :] = torch.rand([num_samples,4])
  calib_points[neg_mask, 0] = 1
  calib_points[neg_mask, 1] = (2*1.1 * calib_points[neg_mask, 1]) - 1.1
  calib_points[neg_mask, 2] = (2*1.1 * calib_points[neg_mask, 2]) - 1.1
  calib_points[neg_mask, 3] = (2*torch.pi * calib_points[neg_mask, 3]) - torch.pi

  with torch.no_grad():
      model_results = model({'coords': dataset.dynamics.coord_to_input(calib_points.to(device))})
      values = dataset.dynamics.io_to_value(model_results['model_in'].detach(), model_results['model_out'].squeeze(dim=-1).detach())

  pos_mask = values > 0
  num_pos = torch.sum(pos_mask)
  neg_mask = ~pos_mask


# simulate all N points with the optimal control
model_results = model({'coords': dataset.dynamics.coord_to_input(calib_points.to(device))})
dvs = dataset.dynamics.io_to_dv(model_results['model_in'], model_results['model_out'].squeeze(dim=-1))
dvdt = dvs[:, 0]
dvds = dvs[:, 1:]
ctrl_trajs = dataset.dynamics.optimal_control(calib_points[:, 1:].to(device), dvds.to(device))


dt = 0.01

T = 100 # simulation horizon of 1 second

worst_cost = dataset.dynamics.boundary_fn(calib_points[:,1:])
traj_hist = [calib_points[:,1:].clone().detach()]
for _ in range(T):
  model_results = model({'coords': dataset.dynamics.coord_to_input(calib_points.to(device))})
  dvs = dataset.dynamics.io_to_dv(model_results['model_in'], model_results['model_out'].squeeze(dim=-1))
  dvdt = dvs[:, 0]
  dvds = dvs[:, 1:]
  ctrl_trajs = dataset.dynamics.optimal_control(calib_points[:, 1:].to(device), dvds.to(device))
  dsdt = dataset.dynamics.dsdt(calib_points[:, 1:].to(device), ctrl_trajs, None)
  calib_points[:, 1:] += dt * dsdt
  calib_points[:, 1:] = dataset.dynamics.equivalent_wrapped_state(calib_points[:, 1:])
  traj_hist.append(calib_points[:, 1:].clone().detach())
  worst_cost = torch.minimum(worst_cost, dataset.dynamics.boundary_fn(calib_points[:,1:]))


traj_hist = torch.stack(traj_hist)
k = torch.sum(worst_cost < 0)
print('Number of points:', N, "Number of outliers: ", k.item())


Let's visualize the trajectories we just simulated!

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)
# add circle with radius 0.5
circle = plt.Circle((0, 0), 0.5, color='r', fill=False)
plt.gca().add_artist(circle)

for i in range(N):
    plt.scatter(traj_hist[:, i, 0], traj_hist[:, i, 1])
print(traj_hist[:, 0, 0].size())

As you can see, the resulting safety filter isn't perfect ($k > 0$). This means that there is some mismatch between a state being in the zero-superlevel set of our learned safety value function and that state actually being safe by following the resulting safe policy.


Following [1], we can chose $\beta$ such that

$\sum_{i=0}^k \binom Ni \epsilon^i (1 - \epsilon)^{N-i} \leq \beta$

for some empircal safety violation threshold $\epsilon$.

Then, with $(1-\beta)$ probability

$P_{x \in \mathcal{S}}(V(x) \leq 0) \leq \epsilon$

where $\mathcal{S}$ denotes the zero-superlevel set of our learned value function. This expression denotes the fraction of states within $\mathcal{S}$ that are actually unsafe is bounded above by $\epsilon$ with probability $(1 - \beta)$.


One way to select a suitable choice of parameters $\epsilon$ and $\beta$ is to fix $\epsilon$ and select $\beta = \sum_{i=0}^k \binom Ni \epsilon^i (1 - \epsilon)^{N-i}$


The proof for this result is in Appendix B of https://sia-lab-git.github.io/Verification_of_Neural_Reachable_Tubes.pdf


In [ ]:

def binomial_sum(k, eps, N):
    total = 0.0
    binomial_coeff = 1  # binomial(N, 0) = 1
    eps_power = 1  # eps^0 = 1
    one_minus_eps_power = (1 - eps) ** N  # (1 - eps)^N initially

    for i in range(k + 1):
        # Compute the i-th term: binomial_coeff * eps^i * (1 - eps)^(N - i)
        total += binomial_coeff * eps_power * one_minus_eps_power

        # Update binomial coefficient for next i
        if i < k:
            binomial_coeff *= (N - i) / (i + 1)  # Compute binomial(N, i+1) from binomial(N, i)

        # Update powers of eps and (1 - eps)
        eps_power *= eps  # eps^i -> eps^(i+1)
        one_minus_eps_power *= (1 - eps)  # (1 - eps)^(N - i) -> (1 - eps)^(N - (i+1))

    return total


# TODO: define your own safety-violation rate!
eps = 0.001 # safety-violation rate

beta = binomial_sum(k, eps, N)
print('epsilon:', eps, 'beta:', beta)

**TODO 3.1** Plot on the y-axis $(1 - \beta)$ and $log(\epsilon)$ on the x-axis. Keep $N$ fixed and include this value in the plot title. Do this for 5 values of $log(\epsilon)$

**TODO 3.2** Plot on the y-axis $(1 - \beta)$ and $N$ on the x-axis. Keep $\epsilon$ fixed and include this value in the plot title. Do this for 5 values of $N$

**TODO 3.3** In at least 3-4 sentences, describe your results.

**TODO 3.4** For the dubins car, Deepreach can produce a high quality BRT. If our BRT quality degraded, how would you expect the statistical assurances to change?

In other words, how do we expect the confidence (i.e., $(1-\beta)$) to change for a given safety threshold $\epsilon$ if we encountered many states where the learned value $V(x)$ are empirically unsafe (high number of outliers k).

**Deliverables**

1. Conformal Prediction
Q1.1
Q1.2
Q1.3

2. Uncertainty-aware Failure Detection in Robotic Jenga
Q2.1
Q2.2
Q2.3
Q2.4
Q2.5
Q2.6
Q2.7

3. Conformal Verification of Safe Sets
Q3.1
Q3.2
Q3.3
Q3.4